# Week 9 Notebook: VoltCore Energy — Network Optimization (Parts 3 and 4)

*OMAIB · Module 1 · Week 9*

*© H. Sami Karaca. Questrom School of Business, Boston University.
Online Master of Science in Artificial Intelligence in Business.
Licensed for use in this course only; not for redistribution.*

**What this notebook is.** This is a practice notebook: it is not submitted and does not affect your grade. It runs in two acts. **Act 1** re-runs the Part 3 order book from this week's readings — same tables, same answer — plus the sensitivity report those pages taught you to read. Nothing conceptually new happens in Act 1; that's the point. Seeing the machinery produce numbers you've already verified is what earns trust for the scale where you can't. **Act 2** builds the Part 4 network, then stress-tests it against three disruptions.

**How to read every output.** Solution status, variable values, objective value — then, new this week, the **sensitivity report**: shadow prices, allowable ranges, reduced costs, and the exact point where a guarantee ends.

# Environment Setup

Before running any cells, select **File › Save a copy in Drive** so your work saves to your own Google Drive. This notebook loads everything it needs automatically — there's nothing for you to upload. Run the cells in order. This version uses the libraries already in your session. Results may vary with the runtime; use your own output in your answers.

In [ ]:
# Install the Pyomo optimization modeling package (+ tabulate for output tables)
!pip install -q pyomo tabulate

# Install the optimization engine (solution algorithms)
# GLPK linear-program solver: https://www.gnu.org/software/glpk/
!apt-get install -y -qq glpk-utils

import pyomo.environ as pyo
from pyomo.opt import SolverFactory
from tabulate import tabulate as tb

## Act 1 — Part 3: The Order Book

The Georgia plant plans a fuller quarter: three pack types across five production lines, with **firm customer orders that are floors, not ceilings** — the plant may build more of any pack, never less.

| Pack | Contribution per pack | Firm orders already accepted |
| --- | --- | --- |
| Standard-Range (SR) | \$1,650 | 10,000 |
| Long-Range (LR) | \$2,000 | 3,000 |
| Performance (PF) | \$2,250 | 5,000 |

| Resource | Hours available | SR (hrs/pack) | LR (hrs/pack) | PF (hrs/pack) |
| --- | --- | --- | --- | --- |
| Electrode coating | 6,000 | 0.15 | 0.20 | 0.25 |
| Pack assembly | 7,500 | 0.25 | 0.20 | 0.15 |
| Cell formation | 7,500 | 0.25 | 0.20 | 0.15 |
| Finishing | 6,000 | 0.10 | 0.20 | 0.25 |
| Testing | 7,500 | 0.25 | 0.20 | 0.20 |

**Goal: maximize this quarter's total contribution.** Run the cell, then read: status → plan → contribution.

In [ ]:
# Initialize the model by defining a pyomo instance
model = pyo.ConcreteModel()

# ------------------------------------
# Decision Variables                  |
# ------------------------------------
# SR = Standard-Range packs, LR = Long-Range packs, PF = Performance packs
model.SR = pyo.Var(bounds=(0, None))
model.LR = pyo.Var(bounds=(0, None))
model.PF = pyo.Var(bounds=(0, None))
SR = model.SR
LR = model.LR
PF = model.PF

# ------------------------------------
# Objective Function                  |
# ------------------------------------
# Maximize this quarter's total contribution
model.obj = pyo.Objective(expr = 1650*SR + 2000*LR + 2250*PF, sense = pyo.maximize)

# ------------------------------------
# Constraints                         |
# ------------------------------------
# One constraint per production line (hours used <= hours available)
model.Coating       = pyo.Constraint(expr = 0.15*SR + 0.20*LR + 0.25*PF <= 6000, doc='Electrode coating hours')
model.Assembly      = pyo.Constraint(expr = 0.25*SR + 0.20*LR + 0.15*PF <= 7500, doc='Pack assembly hours')
model.CellFormation = pyo.Constraint(expr = 0.25*SR + 0.20*LR + 0.15*PF <= 7500, doc='Cell formation hours')
model.Finishing     = pyo.Constraint(expr = 0.10*SR + 0.20*LR + 0.25*PF <= 6000, doc='Finishing hours')
model.Testing       = pyo.Constraint(expr = 0.25*SR + 0.20*LR + 0.20*PF <= 7500, doc='Testing hours')

# Firm orders are FLOORS: the plant must build at least these quantities
model.FloorSR = pyo.Constraint(expr = SR >= 10000, doc='Standard-Range firm orders')
model.FloorLR = pyo.Constraint(expr = LR >= 3000,  doc='Long-Range firm orders')
model.FloorPF = pyo.Constraint(expr = PF >= 5000,  doc='Performance firm orders')

# ------------------------------------
# Solve the Model                     |
# ------------------------------------
# sens.txt below holds the full report; this line brings the shadow prices back
# as Python numbers so we can print tidy labeled tables after the solve
model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

opt = SolverFactory('glpk')
opt.options['ranges'] = 'sens_part3.txt'   # full sensitivity report, incl. allowable ranges
results = opt.solve(model, symbolic_solver_labels=True)

# ------------------------------------
# Print Results                       |
# ------------------------------------
print('--------------- RESULTS: PART 3 --------------- \n')
print('Solution status        :', results.solver.termination_condition)
print('Standard-Range packs   :', f'{round(SR()):,}')
print('Long-Range packs       :', f'{round(LR()):,}')
print('Performance packs      :', f'{round(PF()):,}')
print('Quarterly contribution :', f'${round(model.obj()):,}')

In [ ]:
# Line usage and shadow prices at the optimal plan
usage = [['Electrode coating', 0.15*SR() + 0.20*LR() + 0.25*PF(), 6000, model.dual[model.Coating]],
         ['Pack assembly',     0.25*SR() + 0.20*LR() + 0.15*PF(), 7500, model.dual[model.Assembly]],
         ['Cell formation',    0.25*SR() + 0.20*LR() + 0.15*PF(), 7500, model.dual[model.CellFormation]],
         ['Finishing',         0.10*SR() + 0.20*LR() + 0.25*PF(), 6000, model.dual[model.Finishing]],
         ['Testing',           0.25*SR() + 0.20*LR() + 0.20*PF(), 7500, model.dual[model.Testing]]]

print(tb([[n, f'{u:,.0f}', f'{c:,}', f'${d:,.0f} / hour'] for n, u, c, d in usage],
         headers=['Production line', 'Hours used', 'Hours available', 'Shadow price'],
         tablefmt='fancy_grid'))

# The firm-order floors have shadow prices too -- note the sign
floors = [['Standard-Range floor', 10000, SR(), model.dual[model.FloorSR]],
          ['Long-Range floor',     3000,  LR(), model.dual[model.FloorLR]],
          ['Performance floor',    5000,  PF(), model.dual[model.FloorPF]]]
print()
fmt = lambda d: (f'-${abs(d):,.0f} / pack' if d < -1e-9 else f'${d:,.0f} / pack')
print(tb([[n, f'{req:,}', f'{round(built):,}', fmt(d)] for n, req, built, d in floors],
         headers=['Firm-order commitment', 'Packs required', 'Packs built', 'Shadow price'],
         tablefmt='fancy_grid'))

**Check your reading.** The status should be `optimal`, the plan **17,500 Standard-Range / 10,625 Long-Range / 5,000 Performance** — the Performance pack at exactly its contractual minimum — and contribution **\$61,375,000**, matching the case pages to the dollar. In the tables: coating and testing are binding, with shadow prices of **\$8,500** and **\$1,500** per hour; assembly, cell formation, and finishing carry slack and price at zero. And the Performance floor's shadow price is **negative \$175 per pack** — each pack the contract forces costs \$175 in better uses of the bottleneck, exactly as the readings' deep dive showed.

Open `sens_part3.txt` (displayed below) and find two more numbers from the readings: the coating line's allowable increase of about **583 hours** — where the \$8,500 guarantee ends — and the testing line's **250 hours**, the range behind the Read the Range rehearsal.

In [ ]:
# Display the full GLPK sensitivity report for Part 3
with open('sens_part3.txt', 'r') as f:
    print(f.read())

## Act 2 — Part 4: The Global Network

*The block below is the full case description — this is what you copy into your GenAI tool for Ask 2 on the Colab page, before running any cells.*

---

VoltCore operates **four plants** serving **four regional markets**, and must decide how much each plant produces and where each plant ships, to **maximize total annual profit contribution**.

| Plant | Annual capacity (GWh) | Production cost (\$/kWh) |
| --- | --- | --- |
| Coweta County, Georgia (USA) | 10 | 58.17 |
| Sparks, Nevada (USA) | 20 | 53.67 |
| Gdańsk (Poland) | 12 | 63.92 |
| Ulsan (South Korea) | 25 | 39.00 |

| Market | Max demand (GWh) | Selling price (\$/kWh) |
| --- | --- | --- |
| Atlanta (US Southeast) | 15 | 92.50 |
| São Paulo (Latin America) | 7.5 | 101.83 |
| Rotterdam (European hub) | 20 | 96.33 |
| Shanghai (Asia) | 10 | 104.42 |

Transport + tariff cost (\$/kWh), by plant (row) and market (column):

| From \\ To | Atlanta | São Paulo | Rotterdam | Shanghai |
| --- | --- | --- | --- | --- |
| Georgia | 0.00 | 20.38 | 15.13 | 35.75 |
| Nevada | 7.50 | 27.50 | 22.25 | 29.75 |
| Gdańsk | 15.25 | 21.00 | 0.00 | 20.88 |
| Ulsan | 35.75 | 30.75 | 25.25 | 9.88 |

The profit contribution on any lane, per GWh, equals the market's selling price minus the plant's production cost minus that lane's transport-and-tariff cost. A plant's total output cannot exceed its capacity; total shipments into a market cannot exceed its demand. VoltCore is **not** obligated to fully serve every market. (One GWh sold at \$1/kWh yields \$1 million, so \$/kWh and \$M/GWh are the same number — the objective below is in \$M.)

Each decision variable $x_{ij}$ is the volume (in GWh) produced at plant $i$ and shipped to market $j$, where $i$ runs 1–4 (1 = Georgia, 2 = Nevada, 3 = Gdańsk, 4 = Ulsan) and $j$ runs 1–4 (1 = Atlanta, 2 = São Paulo, 3 = Rotterdam, 4 = Shanghai). In code, $x_{23}$ — Nevada to Rotterdam — is written `x23`. The next cell defines the whole network as a reusable function — the scenarios later change one number each and call it again.

In [ ]:
def solve_network(ulsan_capacity=25, rotterdam_demand=20, ua_tariff=35.75,
                  gdansk_capacity=12, ranges_file=None):
    """Build and solve the VoltCore Part 4 network. Defaults are the base case;
    each scenario changes exactly one argument."""

    model = pyo.ConcreteModel()

    # ------------------------------------
    # Decision Variables (GWh per lane)   |
    # ------------------------------------
    for name in ['x11','x12','x13','x14','x21','x22','x23','x24',
                 'x31','x32','x33','x34','x41','x42','x43','x44']:
        setattr(model, name, pyo.Var(bounds=(0, None)))
    x11, x12, x13, x14 = model.x11, model.x12, model.x13, model.x14   # Georgia ->
    x21, x22, x23, x24 = model.x21, model.x22, model.x23, model.x24   # Nevada ->
    x31, x32, x33, x34 = model.x31, model.x32, model.x33, model.x34   # Gdansk ->
    x41, x42, x43, x44 = model.x41, model.x42, model.x43, model.x44   # Ulsan ->

    # ------------------------------------
    # Objective Function ($M)             |
    # ------------------------------------
    # market price x volume in - plant cost x volume out - lane transport+tariff
    model.obj = pyo.Objective(expr =
          92.50*(x11 + x21 + x31 + x41)      # Atlanta selling price
        + 101.83*(x12 + x22 + x32 + x42)     # Sao Paulo selling price
        +  96.33*(x13 + x23 + x33 + x43)     # Rotterdam selling price
        + 104.42*(x14 + x24 + x34 + x44)     # Shanghai selling price
        -  58.17*(x11 + x12 + x13 + x14)     # Georgia production cost
        -  53.67*(x21 + x22 + x23 + x24)     # Nevada production cost
        -  63.92*(x31 + x32 + x33 + x34)     # Gdansk production cost
        -  39.00*(x41 + x42 + x43 + x44)     # Ulsan production cost
        - (  0.00*x11 + 20.38*x12 + 15.13*x13 +     35.75*x14      # Georgia lanes
           + 7.50*x21 + 27.50*x22 + 22.25*x23 +     29.75*x24      # Nevada lanes
           +15.25*x31 + 21.00*x32 +  0.00*x33 +     20.88*x34      # Gdansk lanes
           +ua_tariff*x41 + 30.75*x42 + 25.25*x43 +  9.88*x44)     # Ulsan lanes
        , sense = pyo.maximize)

    # ------------------------------------
    # Constraints                         |
    # ------------------------------------
    # Plant capacities (GWh)
    model.CapGeorgia = pyo.Constraint(expr = x11 + x12 + x13 + x14 <= 10)
    model.CapNevada  = pyo.Constraint(expr = x21 + x22 + x23 + x24 <= 20)
    model.CapGdansk  = pyo.Constraint(expr = x31 + x32 + x33 + x34 <= gdansk_capacity)
    model.CapUlsan   = pyo.Constraint(expr = x41 + x42 + x43 + x44 <= ulsan_capacity)

    # Market demand ceilings (GWh) -- ceilings, not obligations
    model.DemAtlanta   = pyo.Constraint(expr = x11 + x21 + x31 + x41 <= 15)
    model.DemSaoPaulo  = pyo.Constraint(expr = x12 + x22 + x32 + x42 <= 7.5)
    model.DemRotterdam = pyo.Constraint(expr = x13 + x23 + x33 + x43 <= rotterdam_demand)
    model.DemShanghai  = pyo.Constraint(expr = x14 + x24 + x34 + x44 <= 10)

    # ------------------------------------
    # Solve the Model                     |
    # ------------------------------------
    model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)   # bring shadow prices back for the tables
    opt = SolverFactory('glpk')
    if ranges_file:
        opt.options['ranges'] = ranges_file
    results = opt.solve(model, symbolic_solver_labels=True)
    return model, results

# ------------------------------------
# Base run                             |
# ------------------------------------
base, results = solve_network(ranges_file='sens_part4.txt')

print('--------------- RESULTS: PART 4 (BASE CASE) --------------- \n')
print('Solution status           :', results.solver.termination_condition)
print('Gross annual contribution :', f'${base.obj():,.1f}M')

In [ ]:
# Shipment plan (GWh) and plant utilization
def shipment_tables(m):
    g = lambda v: 0.0 if abs(v) < 1e-9 else round(v, 2)   # tidy tiny float noise
    plan = [['From \\ To', 'Atlanta', 'Sao Paulo', 'Rotterdam', 'Shanghai'],
            ['Georgia', g(m.x11()), g(m.x12()), g(m.x13()), g(m.x14())],
            ['Nevada',  g(m.x21()), g(m.x22()), g(m.x23()), g(m.x24())],
            ['Gdansk',  g(m.x31()), g(m.x32()), g(m.x33()), g(m.x34())],
            ['Ulsan',   g(m.x41()), g(m.x42()), g(m.x43()), g(m.x44())]]
    print(tb(plan, headers='firstrow', tablefmt='fancy_grid'))

    caps = [['Georgia', m.x11()+m.x12()+m.x13()+m.x14(), m.CapGeorgia.upper()],
            ['Nevada',  m.x21()+m.x22()+m.x23()+m.x24(), m.CapNevada.upper()],
            ['Gdansk',  m.x31()+m.x32()+m.x33()+m.x34(), m.CapGdansk.upper()],
            ['Ulsan',   m.x41()+m.x42()+m.x43()+m.x44(), m.CapUlsan.upper()]]
    print()
    print(tb([[p, f'{u:,.1f}', f'{c:,.1f}'] for p, u, c in caps],
             headers=['Plant', 'GWh produced', 'Capacity (GWh)'], tablefmt='fancy_grid'))

    dems = [['Atlanta',   m.x11()+m.x21()+m.x31()+m.x41(), m.DemAtlanta.upper()],
            ['Sao Paulo', m.x12()+m.x22()+m.x32()+m.x42(), m.DemSaoPaulo.upper()],
            ['Rotterdam', m.x13()+m.x23()+m.x33()+m.x43(), m.DemRotterdam.upper()],
            ['Shanghai',  m.x14()+m.x24()+m.x34()+m.x44(), m.DemShanghai.upper()]]
    print()
    print(tb([[mk, f'{g(u):,.1f}', f'{c:,.1f}'] for mk, u, c in dems],
             headers=['Market', 'GWh received', 'Demand ceiling (GWh)'], tablefmt='fancy_grid'))

shipment_tables(base)

In [ ]:
# Shadow prices: what one more GWh is worth, plant by plant and market by market
plants  = [['Georgia capacity',  base.dual[base.CapGeorgia]],
           ['Nevada capacity',   base.dual[base.CapNevada]],
           ['Gdansk capacity',   base.dual[base.CapGdansk]],
           ['Ulsan capacity',    base.dual[base.CapUlsan]]]
markets = [['Atlanta demand',    base.dual[base.DemAtlanta]],
           ['Sao Paulo demand',  base.dual[base.DemSaoPaulo]],
           ['Rotterdam demand',  base.dual[base.DemRotterdam]],
           ['Shanghai demand',   base.dual[base.DemShanghai]]]

print(tb([[n, f'${d:,.2f}M / GWh'] for n, d in plants],
         headers=['Plant constraint', 'Shadow price'], tablefmt='fancy_grid'))
print()
print(tb([[n, f'${d:,.2f}M / GWh'] for n, d in markets],
         headers=['Market constraint', 'Shadow price'], tablefmt='fancy_grid'))

**Check your reading.** Gross contribution should be **\$1,935.8M** — the \$1,936M you met on the topic pages, before the pages rounded. The routing should match the reading's reveal exactly: Georgia→Atlanta 10; Nevada→Atlanta 5 and →São Paulo 0.5; Gdańsk→Rotterdam 12; Ulsan→São Paulo 7, →Rotterdam 8, →Shanghai 10. Georgia, Gdańsk, and Ulsan run at capacity; **Nevada runs 5.5 of 20** — and its shadow price reads exactly **zero**, which is what slack looks like in a report. Gdańsk's **\$11.75M/GWh** is the plant-side star; **Shanghai's \$44.12M/GWh** the market side's. All four markets are fully served — the permission to leave demand unserved exists, and the base plan doesn't use it.

In `sens_part4.txt` below, find the range the readings warned about: Gdańsk's \$11.75M holds for only about **0.5 GWh** of expansion. One more number from the readings hides in the report's column table: the unused Ulsan-to-Atlanta lane's reduced cost — about **\$25 per kWh**, the price of entry.

In [ ]:
# Display the full GLPK sensitivity report for Part 4
with open('sens_part4.txt', 'r') as f:
    print(f.read())

In [ ]:
# Reference: the solved network at a glance
# Line width = GWh shipped in the base solution; color = transport+tariff cost.
# Unused lanes are thin and dashed -- notice where Nevada's lanes sit.
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

plants_pos  = {'Georgia': 3, 'Nevada': 2, 'Gdansk': 1, 'Ulsan': 0}
markets_pos = {'Atlanta': 3, 'Sao Paulo': 2, 'Rotterdam': 1, 'Shanghai': 0}
tariffs = {('Georgia','Atlanta'):0.00, ('Georgia','Sao Paulo'):20.38, ('Georgia','Rotterdam'):15.13, ('Georgia','Shanghai'):35.75,
           ('Nevada','Atlanta'):7.50,  ('Nevada','Sao Paulo'):27.50,  ('Nevada','Rotterdam'):22.25,  ('Nevada','Shanghai'):29.75,
           ('Gdansk','Atlanta'):15.25, ('Gdansk','Sao Paulo'):21.00,  ('Gdansk','Rotterdam'):0.00,   ('Gdansk','Shanghai'):20.88,
           ('Ulsan','Atlanta'):35.75,  ('Ulsan','Sao Paulo'):30.75,   ('Ulsan','Rotterdam'):25.25,   ('Ulsan','Shanghai'):9.88}
flows = {('Georgia','Atlanta'):base.x11(), ('Georgia','Sao Paulo'):base.x12(), ('Georgia','Rotterdam'):base.x13(), ('Georgia','Shanghai'):base.x14(),
         ('Nevada','Atlanta'):base.x21(),  ('Nevada','Sao Paulo'):base.x22(),  ('Nevada','Rotterdam'):base.x23(),  ('Nevada','Shanghai'):base.x24(),
         ('Gdansk','Atlanta'):base.x31(),  ('Gdansk','Sao Paulo'):base.x32(),  ('Gdansk','Rotterdam'):base.x33(),  ('Gdansk','Shanghai'):base.x34(),
         ('Ulsan','Atlanta'):base.x41(),   ('Ulsan','Sao Paulo'):base.x42(),   ('Ulsan','Rotterdam'):base.x43(),   ('Ulsan','Shanghai'):base.x44()}

fig, ax = plt.subplots(figsize=(9, 5))
norm = mcolors.Normalize(vmin=0, vmax=36)
for (p, m), t in tariffs.items():
    f = flows[(p, m)]
    if f > 1e-6:
        ax.plot([0, 1], [plants_pos[p], markets_pos[m]], color=cm.RdYlGn_r(norm(t)),
                linewidth=1 + f, alpha=0.9, solid_capstyle='round')
    else:
        ax.plot([0, 1], [plants_pos[p], markets_pos[m]], color=cm.RdYlGn_r(norm(t)),
                linewidth=0.7, alpha=0.35, linestyle='--')
for p, y in plants_pos.items():
    ax.text(-0.03, y, p, ha='right', va='center', fontsize=11, fontweight='bold')
for m, y in markets_pos.items():
    ax.text(1.03, y, m, ha='left', va='center', fontsize=11, fontweight='bold')
ax.set_xlim(-0.35, 1.35); ax.set_ylim(-0.5, 3.5); ax.axis('off')
ax.set_title('Base-case flows: width = GWh shipped, color = transport + tariff cost\n(dashed = lane unused in the optimal plan)')
fig.colorbar(cm.ScalarMappable(norm=norm, cmap='RdYlGn_r'), ax=ax, label='$/kWh')
plt.show()

## The Three Scenarios — Predict, Run, Compare

Three disruptions hit the network. For each one: **predict first** from the base solution and its sensitivity report — does the plan survive, where does volume shift, roughly what does it cost — then run the cell and compare. If you prefer structured notes, copy this table into your scratch space:

| Scenario | My prediction (\$ impact) | Model result | Did the guardrails hold? |
| --- | --- | --- | --- |
| A — Fire at Ulsan | | | |
| B — Rotterdam goes dark | | | |
| C — The tariff headline | | | |

### Scenario A — Fire at Ulsan
An overnight fire takes roughly 20% of Ulsan's capacity offline: 25 GWh falls to 20 for the quarter. Ulsan is binding with a shadow price of \$11.42M/GWh — but a shadow price is only guaranteed inside its allowable range, and this is a 5 GWh cut. Does the range cover it? Predict before you run.

In [ ]:
# Scenario A: Ulsan capacity 25 -> 20 (everything else unchanged)
mA, rA = solve_network(ulsan_capacity=20)

print('--------------- SCENARIO A: FIRE AT ULSAN --------------- \n')
print('Solution status           :', rA.solver.termination_condition)
print('Gross annual contribution :', f'${mA.obj():,.1f}M')
print('Change vs base case       :', f'${mA.obj() - base.obj():,.1f}M')
print()
shipment_tables(mA)

### Scenario B — Rotterdam Goes Dark
A port strike closes the Rotterdam hub: demand effectively drops to zero. Twenty GWh of shipments — Gdańsk's entire output and 8 from Ulsan — have nowhere to land, and every other market's ceiling is already binding. Predict: who is hit hardest, and can anything be redirected?

In [ ]:
# Scenario B: Rotterdam demand 20 -> 0 (everything else unchanged)
mB, rB = solve_network(rotterdam_demand=0)

print('--------------- SCENARIO B: ROTTERDAM GOES DARK --------------- \n')
print('Solution status           :', rB.solver.termination_condition)
print('Gross annual contribution :', f'${mB.obj():,.1f}M')
print('Change vs base case       :', f'${mB.obj() - base.obj():,.1f}M')
print()
shipment_tables(mB)

### Scenario C — The Tariff Headline
A trade ruling raises the Ulsan-to-Atlanta transport-and-tariff cost by 30%: \$35.75 becomes \$46.48 per kWh. The business press calls it "a direct hit to VoltCore's global network." Before you run: check the base shipment table — how much volume does that lane actually carry?

In [ ]:
# Scenario C: Ulsan->Atlanta tariff +30% (35.75 -> 46.475, everything else unchanged)
mC, rC = solve_network(ua_tariff=35.75 * 1.30)

print('--------------- SCENARIO C: THE TARIFF HEADLINE --------------- \n')
print('Solution status           :', rC.solver.termination_condition)
print('Gross annual contribution :', f'${mC.obj():,.1f}M')
print('Change vs base case       :', f'${mC.obj() - base.obj():,.1f}M')

**Check your reading.** A: the plan survives but pays — about **−\$57.1M**, exactly 5 × \$11.42M: the allowable range covered the whole cut, so shadow-price arithmetic priced the fire precisely, with Nevada's slack absorbing what it profitably can. B: a genuine catastrophe — about **−\$639.8M**, nearly two-fifths of the network's demand gone with nowhere to redirect it (watch Gdańsk's row go dark). C: exactly **\$0.0M** — the loudest headline in the set, worth nothing, because the optimal plan never used that lane. The sensitivity report told you each of these stories before you ran a thing: slack absorbs, ranges tell you when the arithmetic is exact (A) and when it expires (B, and D below), and an unused lane's cost can rise forever.

### Optional: Scenario D — The Guardrail's End
Gdańsk's shadow price says \$11.75M per GWh — valid for about 0.5 GWh. What happens if you expand by 2 GWh anyway? Shadow-price arithmetic promises 2 × \$11.75M = \$23.5M. Run it.

In [ ]:
# Optional Scenario D: Gdansk capacity 12 -> 14
# (a 2 GWh expansion; only the first ~0.5 GWh sits inside the allowable range)
mD, rD = solve_network(gdansk_capacity=14)

print('--------------- SCENARIO D: GDANSK +2 GWH --------------- \n')
print('Gross annual contribution :', f'${mD.obj():,.1f}M')
print('Realized gain vs base     :', f'${mD.obj() - base.obj():,.1f}M')
print('Shadow-price arithmetic   : 2 x $11.75M = $23.5M')
print()
print('The gap between those two numbers is the allowable range ending in real')
print('time: only the first ~0.5 GWh earns $11.75M; past it, Rotterdam saturates.')

---
## Before You Leave

For your **VoltCore Audit Log** assignment on Blackboard (Entry 2, both rows): set your GenAI tool's Part 3 and Part 4 answers beside the verified plans above — did it treat the firm orders as minimums? did it handle the lane-specific tariffs and the demand ceilings as given? Your scenario reasoning is not collected here; it's the raw material for this week's simulation activity. The notebook itself is not submitted.

*Next week the network meets its hardest question: which of these four plants deserves to stay open at all.*